## Noun-Level Semantic Similarity for Sentence Pair Evaluation Using Token Embeddings

When comparing token-level nouns from each sentence pair, the process compares each noun in the ground-truth sentence to all nouns in the corresponding generated sentence, and then computes a similarity for the pair of sentences based on these token-to-token similarities.

### What pairs are being compared?
For a sentence pair (ground-truth vs. generated), we:
1. Extract nouns (or noun phrases) from both sentences.
2. Embed each noun using a SentenceTransformer (e.g., all-MiniLM-L6-v2).
3. Compute cosine similarity between every noun in the ground-truth sentence and every noun in the generated sentence, forming a similarity matrix.
4. For each ground-truth noun, take the most similar noun in the generated sentence (max similarity per row).
5. Average those max similarities to yield a single score per sentence pair.

### What is UMAP doing?
- UMAP (Uniform Manifold Approximation and Projection) is a dimensionality reduction technique.
- It takes high-dimensional vectors (here, the embeddings of the noun tokens from SentenceTransformer).
- It projects those vectors down to 2 dimensions so they can be visualized in a scatter plot.

##### What do the axes (Dimension 1 and 2) mean?
- They don't have a fixed "real-world" meaning like height or weight.
- They are abstract dimensions computed by UMAP to preserve the local and global structure of the data.
- Points close together in the plot mean their original embeddings are similar.
- Points far apart mean their embeddings are dissimilar.

##### What insights can you get from a UMAP plot of your nouns?
- Clusters: Groups of nouns close together probably have similar meanings or are used in similar contexts.
- Overlap: If nouns from ground truth and generated text cluster together, it means the generated text has nouns semantically similar to the ground truth.
- Separation: If ground truth and generated nouns form separate clusters, the generated nouns may be semantically different or less accurate.

##### Example interpretations:
- If the noun "patient" appears close to other medical terms in both ground truth and generated sets -- good semantic alignment.
- If some generated nouns like "t" or "thing" are far away or isolated -- they may be noise or less relevant.
- If clusters correspond to different topics or contexts (e.g., medical vs. travel nouns), that indicates your data's semantic diversity.


<hr style="height:0.1rem;">

### Imports

In [1]:
import spacy
from sentence_transformers import SentenceTransformer, models, util
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import umap.umap_ as umap

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to D:\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
import warnings
warnings.filterwarnings("ignore")

import logging
from transformers import logging as transformers_logging

# Suppress specific logs from transformers
transformers_logging.set_verbosity_error()

<hr style="height:0.1rem;">

### Function Declarations

In [3]:
def plot_noun_similarity_heatmaps_4sets(
    ground_truth, 
    gen1, 
    gen2, 
    gen3, 
    gen4,
    titles
):
    """
    Extracts nouns from sentence pairs and computes average cosine similarity between
    ground-truth and 4 generated sentence sets, then plots heatmaps of the similarity scores.

    Parameters:
        ground_truth (List[str]): List of ground-truth sentences.
        gen1 (List[str]): First set of generated sentences.
        gen2 (List[str]): Second set of generated sentences.
        gen3 (List[str]): Third set of generated sentences.
        gen4 (List[str]): Fourth set of generated sentences.
    """

    def extract_noun_tokens(sentences):
        noun_lists = []
        for sent in sentences:
            doc = nlp(sent)
            nouns = [token.text for token in doc if token.pos_ in ["NOUN", "PROPN"]]
            noun_lists.append(nouns)
        return noun_lists

    def avg_token_similarity(tokens1, tokens2):
        if not tokens1 or not tokens2:
            return 0.0
        emb1 = model.encode(tokens1, convert_to_tensor=True)
        emb2 = model.encode(tokens2, convert_to_tensor=True)
        sim_matrix = util.cos_sim(emb1, emb2)
        return float(sim_matrix.max(dim=1).values.mean())

    # Extract noun tokens
    gt_nouns = extract_noun_tokens(ground_truth)
    gen_nouns = [extract_noun_tokens(g) for g in [gen1, gen2, gen3, gen4]]

    # Compute similarities for all 4 generated sets
    similarity_scores = [
        [avg_token_similarity(gt, gen) for gt, gen in zip(gt_nouns, gen_set)]
        for gen_set in gen_nouns
    ]

    # Plotting
    num_pairs = len(ground_truth)
    fig, axes = plt.subplots(1, 4, figsize=(20, 6))

    for i in range(4):
        sim_matrix = np.array(similarity_scores[i]).reshape(num_pairs, 1)
        sns.heatmap(sim_matrix, annot=True, cmap="Blues", cbar=False,
                    xticklabels=[f"{titles[i]} (Nouns)"],
                    yticklabels=[f"Pair {j+1}" for j in range(num_pairs)],
                    ax=axes[i])
        axes[i].set_title(titles[i] + " Entity/Noun-Level Similarity")

    plt.tight_layout()
    plt.show()

In [4]:
def plot_token_similarity_heatmaps_per_pair(
    ground_truth, 
    generated_sets, 
    pos_tags=["NOUN", "PROPN"]
):
    """
    For each sentence pair, extract noun tokens from ground-truth and multiple generated sentences,
    compute cosine similarity matrices, and plot heatmaps.

    Parameters:
        ground_truth (List[str]): List of ground-truth sentences.
        generated_sets (List[List[str]]): List containing multiple generated sentence lists (e.g., [gen1, gen2, gen3, gen4]).
        pos_tags (List[str]): POS tags to extract (default: ["NOUN", "PROPN"]).
    """

    def extract_tokens(sentence):
        doc = nlp(sentence)
        return [token.text for token in doc if token.pos_ in pos_tags]

    def plot_similarity_heatmap(tokens1, tokens2, title):
        if not tokens1 or not tokens2:
            print(f"Skipping empty tokens in: {title}")
            return

        emb1 = model.encode(tokens1, convert_to_tensor=True)
        emb2 = model.encode(tokens2, convert_to_tensor=True)

        sim_matrix = util.cos_sim(emb1, emb2).cpu().numpy()

        plt.figure(figsize=(max(6, len(tokens2)*1.2), max(4, len(tokens1)*1.2)))
        sns.heatmap(sim_matrix, annot=True, fmt=".2f", cmap="Blues",
                    xticklabels=tokens2, yticklabels=tokens1)
        plt.title(title)
        plt.xlabel("Generated Tokens")
        plt.ylabel("Ground Truth Tokens")
        plt.tight_layout()
        plt.show()

    for i, gt_sent in enumerate(ground_truth):
        gt_tokens = extract_tokens(gt_sent)

        for j, gen_set in enumerate(generated_sets):
            gen_tokens = extract_tokens(gen_set[i])
            plot_similarity_heatmap(gt_tokens, gen_tokens, f"Set {j+1} - Pair {i+1}")

In [5]:
def extract_nouns(
    sentences, 
    pos_tags=["NOUN", "PROPN"]
):
    all_nouns = []
    for sent in sentences:
        doc = nlp(sent)
        all_nouns.extend([token.text for token in doc if token.pos_ in pos_tags])
    return all_nouns

In [6]:
def extract_noun_texts(sentence):
    doc = nlp(sentence)
    nouns = []
    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"]:
            # Filter out short tokens like 't'
            if len(token.text) > 1:
                nouns.append(token.text)
    return nouns

In [7]:
def compute_similarity_and_plot(
    gt_nouns, 
    gen_nouns, 
    title="Noun Similarity Heatmap"
):
    if not gt_nouns or not gen_nouns:
        print(f"Skipping empty tokens in: {title}")
        return

    emb_gt = model.encode(gt_nouns, convert_to_tensor=True)
    emb_gen = model.encode(gen_nouns, convert_to_tensor=True)

    sim_matrix = util.cos_sim(emb_gt, emb_gen).cpu().numpy()

    plt.figure(figsize=(max(10, len(gen_nouns)*0.5), max(6, len(gt_nouns)*0.5)))
    sns.heatmap(sim_matrix, annot=False, cmap="Blues",
                xticklabels=gen_nouns, yticklabels=gt_nouns)
    plt.title(title)
    plt.xlabel("Generated Nouns")
    plt.ylabel("Ground Truth Nouns")
    plt.tight_layout()
    plt.show()

In [8]:
def plot_all_set_similarities(
    ground_truth, 
    *generated_sets
):
    gt_nouns = extract_nouns(ground_truth)

    for i, gen_sentences in enumerate(generated_sets):
        gen_nouns = extract_nouns(gen_sentences)
        compute_similarity_and_plot(
            gt_nouns, 
            gen_nouns, 
            title=f"Noun-Level Similarity: Ground Truth vs Set {i+1}"
        )

In [9]:
def plot_noun_similarity_heatmap(
    ground_truth, 
    generated_text, 
    title="Noun-Level Similarity Heatmap"
):
    """
    Creates a heatmap of cosine similarities between nouns in the ground truth and generated text sets.

    Args:
        ground_truth (list of str): List of ground-truth sentences.
        generated_text (list of str): List of generated sentences.
        title (str): Title of the heatmap.
    """
    # --- Extract nouns ---
    def extract_nouns(sentences):
        tokens = []
        for sentence in sentences:
            doc = nlp(sentence)
            tokens.extend([token.text for token in doc if token.pos_ in ["NOUN", "PROPN"]])
        return tokens

    gt_nouns = extract_nouns(ground_truth)
    gen_nouns = extract_nouns(generated_text)

    if not gt_nouns or not gen_nouns:
        print(f"Skipping heatmap due to empty noun list in: {title}")
        return

    # --- Encode and compute similarity ---
    emb_gt = model.encode(gt_nouns, convert_to_tensor=True)
    emb_gen = model.encode(gen_nouns, convert_to_tensor=True)
    sim_matrix = util.cos_sim(emb_gt, emb_gen).cpu().numpy()

    # --- Plot heatmap ---
    plt.figure(figsize=(max(10, len(gen_nouns)*0.5), max(6, len(gt_nouns)*0.5)))
    sns.heatmap(sim_matrix, annot=False, cmap="Blues",
                xticklabels=gen_nouns, yticklabels=gt_nouns)
    plt.title(title)
    plt.xlabel("Generated Nouns")
    plt.ylabel("Ground Truth Nouns")
    plt.tight_layout()
    plt.show()

In [10]:
def plot_noun_umap_interactive(
    gt_sentences, 
    gen_sentences,
    title,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
):
    gt_nouns = extract_nouns(gt_sentences)
    gt_emb = model.encode(gt_nouns)
    print("gt_nouns (LEN): ", len(gt_nouns))
    
    gen_nouns = extract_nouns(gen_sentences)
    gen_emb = model.encode(gen_nouns)
    print("gen_nouns (LEN): ", len(gen_nouns))
    print("samples: ", (len(gt_nouns)+len(gen_nouns)))
    
    combined_emb = np.vstack([gt_emb, gen_emb])

    reducer = umap.UMAP(
        n_neighbors=n_neighbors, 
        min_dist=min_dist, 
        random_state=random_state
    )
    embedding = reducer.fit_transform(combined_emb)

    n_gt = len(gt_nouns)
    gt_coords = embedding[:n_gt]
    gen_coords = embedding[n_gt:]

    fig = make_subplots(rows=1, cols=1, subplot_titles=[title])

    fig.add_trace(go.Scatter(
        x=gt_coords[:, 0], y=gt_coords[:, 1],
        mode='markers',
        marker=dict(color='blue', size=10),
        name='Ground Truth',
        showlegend=True
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=gen_coords[:, 0], y=gen_coords[:, 1],
        mode='markers',
        marker=dict(color='red', size=10),
        name='Generated Text',
        showlegend=True
    ), row=1, col=1)

    fig.update_xaxes(title_text='Dimension 1', row=1, col=1)
    fig.update_yaxes(title_text='Dimension 2', row=1, col=1)

    fig.update_layout(height=600, width=600, title_text="UMAP Visualization of Nouns (Ground-Truth vs Generated-Text)")
    fig.show()

In [11]:
def plot_4_umap_subplots_orig(
    gt_sentences, 
    gen_sentences_list, 
    titles,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
):
    sns.set_style("whitegrid")  # or use "white" for no grid

    gt_nouns = extract_nouns(gt_sentences)
    gt_emb = model.encode(gt_nouns)
    
    fig = make_subplots(rows=2, cols=2, subplot_titles=titles)
    
    for i, gen_sentences in enumerate(gen_sentences_list):
        gen_nouns = extract_nouns(gen_sentences)
        gen_emb = model.encode(gen_nouns)
        
        combined_emb = np.vstack([gt_emb, gen_emb])
        
        reducer = umap.UMAP(
            n_neighbors=n_neighbors, 
            min_dist=min_dist, 
            random_state=random_state
        )
        embedding = reducer.fit_transform(combined_emb)
        
        n_gt = len(gt_nouns)
        gt_coords = embedding[:n_gt]
        gen_coords = embedding[n_gt:]
        
        row = (i // 2) + 1
        col = (i % 2) + 1
        
        fig.add_trace(go.Scatter(
            x=gt_coords[:, 0], y=gt_coords[:, 1],
            mode='markers',
            marker=dict(color='blue', size=8, opacity=0.7),
            name='Ground Truth',
            text=gt_nouns,
            hoverinfo='text',
            showlegend=(i==0)
        ), row=row, col=col)
        
        fig.add_trace(go.Scatter(
            x=gen_coords[:, 0], y=gen_coords[:, 1],
            mode='markers',
            marker=dict(color='red', size=8, opacity=0.7),
            name='Generated Text',
            text=gen_nouns,
            hoverinfo='text',
            showlegend=(i==0)
        ), row=row, col=col)
        
        fig.update_xaxes(title_text='Dimension 1', row=row, col=col)
        fig.update_yaxes(title_text='Dimension 2', row=row, col=col)
    
    fig.update_layout(height=900, width=1100, title_text="Part-of-Speech (PoS) tagging (Nouns and PROPN) UMAP Visualization (Ground-Truth vs Generated-Text)")
    fig.show()

In [12]:
def plot_4_umap_subplots1(
    gt_sentences, 
    gen_sentences_list, 
    titles,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
):
    sns.set_style("whitegrid")  # or use "white" for no grid

    gt_nouns = extract_nouns(gt_sentences)
    gt_emb = model.encode(gt_nouns)

    # === Fit UMAP only on gt_emb ===
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        random_state=random_state
    )
    reducer.fit(gt_emb)
    gt_coords = reducer.transform(gt_emb)  # (fit + transform = just for GT)

    fig = make_subplots(rows=2, cols=2, subplot_titles=titles, vertical_spacing=0.12, horizontal_spacing=0.08)

    for i, gen_sentences in enumerate(gen_sentences_list):
        gen_nouns = extract_nouns(gen_sentences)
        gen_emb = model.encode(gen_nouns)

        # Transform gen_emb using fitted UMAP
        gen_coords = reducer.transform(gen_emb)

        row = (i // 2) + 1
        col = (i % 2) + 1

        # Plot Ground Truth (same in every plot)
        fig.add_trace(go.Scatter(
            x=gt_coords[:, 0], y=gt_coords[:, 1],
            mode='markers',
            marker=dict(color='blue', size=8, opacity=0.7),
            name='Ground Truth',
            text=gt_nouns,
            hoverinfo='text',
            showlegend=(i == 0)
        ), row=row, col=col)

        # Plot Generated
        fig.add_trace(go.Scatter(
            x=gen_coords[:, 0], y=gen_coords[:, 1],
            mode='markers',
            marker=dict(color='red', size=8, opacity=0.7),
            name='Generated Text',
            text=gen_nouns,
            hoverinfo='text',
            showlegend=(i == 0)
        ), row=row, col=col)

        fig.update_xaxes(title_text='Dimension 1', row=row, col=col)
        fig.update_yaxes(title_text='Dimension 2', row=row, col=col)

    fig.update_layout(
        height=1000, width=1200, 
        title_text="Part-of-Speech (PoS) tagging (Nouns and PROPN) UMAP Visualization (Ground-Truth vs Generated-Text)",
        font=dict(family="Arial", size=12),
    )    
    fig.show()

In [13]:
def plot_4_umap_subplots(
    gt_sentences, 
    gen_sentences_list, 
    titles,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
):
    sns.set_style("whitegrid")

    gt_nouns = extract_nouns(gt_sentences)
    gt_emb = model.encode(gt_nouns)

    # === Fit UMAP only on gt_emb ===
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        random_state=random_state
    )
    reducer.fit(gt_emb)
    gt_coords = reducer.transform(gt_emb)

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=titles,
        vertical_spacing=0.08,
        horizontal_spacing=0.08
    )

    for i, gen_sentences in enumerate(gen_sentences_list):
        gen_nouns = extract_nouns(gen_sentences)
        gen_emb = model.encode(gen_nouns)
        gen_coords = reducer.transform(gen_emb)

        row = (i // 2) + 1
        col = (i % 2) + 1

        # === Plot Ground Truth points ===
        fig.add_trace(go.Scatter(
            x=gt_coords[:, 0], y=gt_coords[:, 1],
            mode='markers+text',
            marker=dict(color='blue', size=8, opacity=0.6),
            text=gt_nouns,
            textposition='top center',
            textfont=dict(size=12, color='gray'),
            name='Ground Truth',
            hoverinfo='text',
            showlegend=(i == 0)
        ), row=row, col=col)

        # === Plot Generated points ===
        fig.add_trace(go.Scatter(
            x=gen_coords[:, 0], y=gen_coords[:, 1],
            mode='markers+text',
            marker=dict(color='red', size=8, opacity=0.6),
            text=gen_nouns,
            textposition='top center',
            textfont=dict(size=12, color='gray'),
            name='Generated Text',
            hoverinfo='text',
            showlegend=(i == 0)
        ), row=row, col=col)

        fig.update_xaxes(title_text='Dimension 1', row=row, col=col)
        fig.update_yaxes(title_text='Dimension 2', row=row, col=col)

    fig.update_layout(
        height=1600, width=2000, 
        title_text="Part-of-Speech (PoS) tagging (Nouns and PROPN) UMAP Visualization (Ground-Truth vs Generated-Text)",
        font=dict(family="Arial", size=14),
    )    
    fig.show()

In [14]:
def plot_4_umap_subplots3(
    gt_sentences, 
    gen_sentences_list, 
    titles,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
):
    sns.set_style("whitegrid")

    gt_nouns = extract_nouns(gt_sentences)
    gt_emb = model.encode(gt_nouns)

    # Fit UMAP on ground-truth only
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        random_state=random_state
    )
    reducer.fit(gt_emb)
    gt_coords = reducer.transform(gt_emb)

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=titles,
        vertical_spacing=0.12,
        horizontal_spacing=0.08
    )

    for i, gen_sentences in enumerate(gen_sentences_list):
        gen_nouns = extract_nouns(gen_sentences)
        gen_emb = model.encode(gen_nouns)
        gen_coords = reducer.transform(gen_emb)

        row = (i // 2) + 1
        col = (i % 2) + 1

        # Ground Truth points
        fig.add_trace(go.Scatter(
            x=gt_coords[:, 0], y=gt_coords[:, 1],
            mode='markers+text',
            marker=dict(color='blue', size=7, opacity=0.4),
            text=gt_nouns,
            textposition='middle right',
            textfont=dict(
                size=10,
                color='blue',
                family='Arial Black'
            ),
            name='Ground Truth',
            hoverinfo='text',
            showlegend=(i == 0)
        ), row=row, col=col)

        # Generated points
        fig.add_trace(go.Scatter(
            x=gen_coords[:, 0], y=gen_coords[:, 1],
            mode='markers+text',
            marker=dict(color='red', size=7, opacity=0.4),
            text=gen_nouns,
            textposition='middle right',
            textfont=dict(
                size=10,
                color='red',
                family='Arial Black'
            ),
            name='Generated Text',
            hoverinfo='text',
            showlegend=(i == 0)
        ), row=row, col=col)

        fig.update_xaxes(title_text='Dimension 1', row=row, col=col)
        fig.update_yaxes(title_text='Dimension 2', row=row, col=col)

    fig.update_layout(
        height=1400, width=1800, 
        title_text="Part-of-Speech (PoS) tagging (Nouns and PROPN) UMAP Visualization (Ground-Truth vs Generated-Text)",
        font=dict(family="Arial", size=12),
    )

    fig.show()

<hr style="height:0.1rem;">

### Load the Spacy model

In [15]:
nlp = spacy.load("en_core_web_sm")

<hr style="height:0.1rem;">

### Load the base BERT model

##### Apply pooling strategy to get sentence-level embeddings

- [BERT](https://huggingface.co/google-bert/bert-large-uncased) - pretrained on a large corpus of English data in a self-supervised fashion
- [SciBERT](https://huggingface.co/allenai/scibert_scivocab_uncased) - pretrained BERT model trained on scientific text

In [16]:
#word_embedding_model = models.Transformer('bert-large-uncased')
word_embedding_model = models.Transformer('allenai/scibert_scivocab_uncased')
#word_embedding_model = models.Transformer('medicalai/ClinicalBERT')

pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())

model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

<hr style="height:0.1rem;">

### Ground-Truth - Clinical Case B in Paper

In [17]:
CLINICAL_CASE_B_ARTICLE_ID = 'S1130-05582017000100031-1'
CLINICAL_CASE_B_TEXT_REFERENCES = ['edentulism']
CLINICAL_CASE_B_ICD_CODES = ['K08109']
CLINICAL_CASE_B_PATIENT_AGE = '54'
CLINICAL_CASE_B_PATIENT_GENDER = 'female'
CLINICAL_CASE_B_CLINICAL_NOTES = '''A 54-year-old female patient was diagnosed with partial edentulism of both arches and severe alveolar sequestration class III of Sept1 in the edentulous premaxilla region. The treatment plan consists of placing two hydrogel pearls sterilized in an autoclave for tissue expansion of the premaxilla, and a subsequent block bone allograft in the same place. Two weeks after the placement of hydrogel pearls, exposure and removal of them were performed, with placement of medium wound block was possible discharge stereotactic marking, not preformed translithotomy incisions and tension could be performed. Once the healing phase of the block finished, the dental implants were placed in an appropriate position, with very good bone rehabilitation and blocking.'''

In [18]:
SNOMED_CT_ONTOLOGY = """
ICD Code: K08109
Ontology Concept Definition:

Concept: Edentulous (finding) (278650002)
Subclass of: Clinical finding (finding) (404684003), Role group (attribute) (609096000), Has interpretation (attribute) (363713009), Absent (qualifier value) (2667000), Interprets (attribute) (363714003), Tooth presence (observable entity) (278652005), Role group (attribute) (609096000), Finding site (attribute) (363698007), All teeth (body structure) (1162715001)
"""

In [19]:
ground_truth_sentences = sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES)
ground_truth_sentences

['A 54-year-old female patient was diagnosed with partial edentulism of both arches and severe alveolar sequestration class III of Sept1 in the edentulous premaxilla region.',
 'The treatment plan consists of placing two hydrogel pearls sterilized in an autoclave for tissue expansion of the premaxilla, and a subsequent block bone allograft in the same place.',
 'Two weeks after the placement of hydrogel pearls, exposure and removal of them were performed, with placement of medium wound block was possible discharge stereotactic marking, not preformed translithotomy incisions and tension could be performed.',
 'Once the healing phase of the block finished, the dental implants were placed in an appropriate position, with very good bone rehabilitation and blocking.']

<hr style="height:0.1rem;">

### Settings

In [20]:
# Set a value between 0-4
BASELINE_GENERATED_TEXT_INDEX = 0

In [21]:
# Set a value between 0-4
COTSS_GENERATED_TEXT_INDEX = 0

In [22]:
# Set a value between 0-4
COTKG_GENERATED_TEXT_INDEX = 0

In [23]:
# Set a value between 0-4
COTSSKG_GENERATED_TEXT_INDEX = 0

<hr style="height:0.1rem;">

### Baseline

Contains: 
- ICD Codes
- Patient Age
- Patient Gender

In [24]:
### R003 means row #3 in the CSV (includes header row as Row 1)
BASELINE_R003_GENERATED_TEXT = """The 54-year-old female patient presents with complaints of tooth loss. She reports that she has lost several teeth over the past few years, not related to any trauma or injury. She also mentions experiencing difficulty in chewing and occasional discomfort due to the missing teeth. She denies any associated pain, swelling, or signs of infection. She has not sought any dental intervention for the tooth loss until now. Her oral hygiene practices are fair, but she admits to not visiting a dentist regularly. She does not have any known allergies or significant medical history. She does not smoke or consume alcohol. Further evaluation and management are planned to address her tooth loss, coded as K08.109."""
BASELINE_R041_GENERATED_TEXT = """The 54-year-old female patient presents with complaints of tooth loss. She reports that she has lost several teeth over the past few months without any apparent cause such as trauma or decay. She denies any associated pain, swelling, or discomfort. She also denies any changes in her eating habits, oral hygiene, or use of tobacco products. She has no history of periodontal disease or other oral health issues. She is concerned about the aesthetic and functional implications of her tooth loss."""
BASELINE_R068_GENERATED_TEXT = """The patient is a 54-year-old female who presents with complaints of tooth loss. She reports that she has lost several teeth over the past few months without any apparent reason. She denies any associated pain, swelling, or infection. She also denies any recent trauma or injury to the mouth. She does not wear dentures. She has not sought any dental care for this issue yet. She does not have any significant medical history. She does not smoke and drinks alcohol occasionally. She maintains good oral hygiene. Her diet is balanced and she does not have any known allergies."""
BASELINE_R088_GENERATED_TEXT = """The patient is a 54-year-old female who presents with complaints of tooth loss. She reports that she has lost several teeth over the past few months without any apparent reason such as trauma or decay. She denies any associated pain, swelling, or infection. She also denies any changes in her eating habits, oral hygiene, or use of tobacco products. She has not sought any dental care for this issue yet. She is concerned about the impact of this tooth loss on her appearance and ability to chew food properly. Further evaluation and management are needed to address her tooth loss, coded as K08.109."""
BASELINE_R096_GENERATED_TEXT = """The 54-year-old female patient presents with complaints of tooth loss. She reports that she has lost several teeth over the past few months without any apparent cause such as trauma or decay. She denies any associated pain, swelling, or discomfort. She also denies any recent changes in her diet or oral hygiene habits. She is concerned about the aesthetic and functional implications of her tooth loss. She has no known allergies and is currently not on any medications. She does not smoke or drink alcohol. Her past medical history is unremarkable."""

# Add selected sentences to list
BASELINE_GENERATED_TEXTS = [
    BASELINE_R003_GENERATED_TEXT,
    BASELINE_R041_GENERATED_TEXT,
    BASELINE_R068_GENERATED_TEXT,
    BASELINE_R088_GENERATED_TEXT,
    BASELINE_R096_GENERATED_TEXT
]

# Convert Baseline to sentences
baseline_sentences = sent_tokenize(BASELINE_GENERATED_TEXTS[BASELINE_GENERATED_TEXT_INDEX])
baseline_sentences

['The 54-year-old female patient presents with complaints of tooth loss.',
 'She reports that she has lost several teeth over the past few years, not related to any trauma or injury.',
 'She also mentions experiencing difficulty in chewing and occasional discomfort due to the missing teeth.',
 'She denies any associated pain, swelling, or signs of infection.',
 'She has not sought any dental intervention for the tooth loss until now.',
 'Her oral hygiene practices are fair, but she admits to not visiting a dentist regularly.',
 'She does not have any known allergies or significant medical history.',
 'She does not smoke or consume alcohol.',
 'Further evaluation and management are planned to address her tooth loss, coded as K08.109.']

<hr style="height:0.1rem;">

### CoT ICD Code Semantic Search

Contains: 
- ICD Codes
- Patient Age
- Patient Gender
- Similar Clinical Example (10 cases) based on ICD Codes

In [25]:
### R057 means row #57 in the CSV (includes header row as Row 1)
COTSS_R005_GENERATED_TEXT = """A 54-year-old female patient presents with a complaint of discomfort in the oral cavity. Upon examination, it is noted that the patient has partial maxillomandibular edentulism, with signs of generalized chronic periodontal disease. There is also an increase in volume in the glomus floor space, which is bilateral. The patient reports a history of controlled hypertension and chronic kidney disease stage V, which has been managed with hemodialysis three times a week. The patient also reports experiencing dysphagia and an increase in volume in the jawbone area and upper limb. The patient's medical history and physical examination suggest a need for further investigation and treatment."""
COTSS_R044_GENERATED_TEXT = """The patient is a 54-year-old female who presents with complaints of discomfort in the oral cavity. Upon examination, there is evidence of partial maxillomandibular edentulism and generalized chronic periodontal disease. The patient also reports halitosis and an increase in volume in the glomus floor space. The patient's medical history reveals controlled hypertension and chronic kidney disease stage V, which has been managed with hemodialysis three times a week. The patient has been experiencing dysphagia and swelling in the jawbone area and upper limb. The patient is currently undergoing treatment with crystalline sodium penicillin and aztreonam. Further evaluation and management are planned in conjunction with the departments of Infectious Diseases and Internal Medicine."""
COTSS_R057_GENERATED_TEXT = """A 54-year-old female patient presents with complaints of discomfort in the oral cavity. Upon examination, it is noted that the patient has partial maxillomandibular edentulism, with signs of generalized chronic periodontal disease. There is also an increase in volume in the glomus floor space bilaterally. The patient reports a history of controlled hypertension and is currently undergoing hemodialysis three times a week due to chronic kidney disease stage V. The patient also reports experiencing dysphagia and an increase in volume in the jawbone area and upper limb. Further examination reveals sclera iccas and a severe increase in volume with predominance in the bilateral submental and submandibular regions. The patient's mouth opening is limited to approximately 15 mm. The presence of local irritants and halitosis is also noted. The patient is referred for further diagnostic tests and treatment planning."""
COTSS_R074_GENERATED_TEXT = """The patient is a 54-year-old female who presents with complaints of discomfort in the oral cavity. Upon examination, there is evidence of partial maxillomandibular edentulism and generalized chronic periodontal disease. The patient also reports halitosis and an increase in volume in the glomus floor space. The patient's medical history reveals controlled hypertension and chronic kidney disease stage V, which has been managed with hemodialysis three times a week. The patient has been experiencing dysphagia and swelling in the jawbone area and upper limb. The patient is currently undergoing treatment with crystalline sodium penicillin and aztreonam, administered intravenously. The patient's condition is being closely monitored by the departments of Infectious Diseases and Internal Medicine. The patient's condition is currently stable, but the evolution of the disease has been slow and challenging. The patient is scheduled for extraction of the affected teeth and surgical drainage of the sublingual, submandibular, and submental spaces. The patient's condition is expected to improve following the surgical intervention."""
COTSS_R095_GENERATED_TEXT = """A 54-year-old female patient presents with complaints of discomfort in the oral cavity. Upon examination, there is evidence of partial maxillomandibular edentulism and generalized chronic periodontal disease. The patient also reports halitosis and an increase in volume in the glomus floor space. The patient's medical history reveals controlled hypertension and chronic kidney disease stage V, which has been managed with hemodialysis three times a week. The patient has been experiencing dysphagia and swelling in the jawbone area and upper limb. The patient's sclera appears icteric. The patient is currently undergoing treatment with crystalline sodium penicillin and aztreonam, administered intravenously. A computed tomography scan reveals an isodense image in the parapharyngeal space, consistent with increased volume, and a diagnosis of Ludwig's angina is made. The patient's condition remains torpid after 10 days of treatment, and a reassessment leads to the initiation of intravenous vancomycin, which results in significant clinical improvement. After systemic stabilization, extraction of the affected teeth is performed, along with surgical drainage of the sublingual, submandibular, and submental spaces. The patient's condition improves with antibiotic therapy"""

# Add selected sentences to list
COTSS_GENERATED_TEXTS = [
    COTSS_R005_GENERATED_TEXT,
    COTSS_R044_GENERATED_TEXT,
    COTSS_R057_GENERATED_TEXT,
    COTSS_R074_GENERATED_TEXT,
    COTSS_R095_GENERATED_TEXT
]

# Convert CoT ICD Semantic Search to sentences
cotss_sentences = sent_tokenize(COTSS_GENERATED_TEXTS[COTSS_GENERATED_TEXT_INDEX])
cotss_sentences

['A 54-year-old female patient presents with a complaint of discomfort in the oral cavity.',
 'Upon examination, it is noted that the patient has partial maxillomandibular edentulism, with signs of generalized chronic periodontal disease.',
 'There is also an increase in volume in the glomus floor space, which is bilateral.',
 'The patient reports a history of controlled hypertension and chronic kidney disease stage V, which has been managed with hemodialysis three times a week.',
 'The patient also reports experiencing dysphagia and an increase in volume in the jawbone area and upper limb.',
 "The patient's medical history and physical examination suggest a need for further investigation and treatment."]

<hr style="height:0.1rem;">

### CoT Knowledge Graph (KG)

Contains: 
- ICD Codes
- Patient Age
- Patient Gender
- SNOMED-CT Ontology

In [26]:
### R009 means row #9 in the CSV (includes header row as Row 1)
COTKG_R009_GENERATED_TEXT = """History of Present Illness: The patient is a 54-year-old female who presents with a complete absence of teeth. She reports having lost all her teeth over the years due to various dental issues. She has not had any dental prosthetics or implants. She has been managing without teeth but is now finding it increasingly difficult to eat and speak properly. She also reports a decrease in self-confidence due to her appearance."""
COTKG_R028_GENERATED_TEXT = """History of Present Illness: The patient is a 54-year-old female who presents with a complete absence of teeth. She reports having lost all her teeth over the years due to various dental issues. She has not had any dental prosthetics or implants. The patient's edentulous condition has been affecting her ability to chew food properly and has been causing her discomfort. She is seeking options for treatment to improve her oral function and appearance."""
COTKG_R042_GENERATED_TEXT = """History of Present Illness: The 54-year-old female patient presents with a complete absence of teeth. She has been edentulous for some time now. The patient reports no presence of any teeth in the oral cavity. This condition has been interpreted as a clinical finding."""
COTKG_R069_GENERATED_TEXT = """History of Present Illness: The patient is a 54-year-old female who presents with complete tooth loss. She reports that she has been without teeth for several years now. She has not had any dental prosthetics in the past and has been managing with her current condition. She denies any recent changes in her oral health. She does not report any pain, discomfort, or difficulty in eating or speaking. However, she expresses concern about her appearance and is interested in possible solutions for her edentulous condition."""
COTKG_R084_GENERATED_TEXT = """History of Present Illness: The patient is a 54-year-old female who presents with complete tooth loss. She reports that she has been without teeth for several years now. She has not had any dental prosthetics fitted and has been managing with her current condition. She denies any recent changes in her oral health. She does not report any pain, discomfort, or difficulty in eating or speaking. She has not noticed any changes in her facial appearance or speech. She is seeking dental care for the first time in many years."""

# Add selected sentences to list
COTKG_GENERATED_TEXTS = [
    COTKG_R009_GENERATED_TEXT,
    COTKG_R028_GENERATED_TEXT,
    COTKG_R042_GENERATED_TEXT,
    COTKG_R069_GENERATED_TEXT,
    COTKG_R084_GENERATED_TEXT
]

# Convert CoT KG to sentences
cotkg_sentences = sent_tokenize(COTKG_GENERATED_TEXTS[COTKG_GENERATED_TEXT_INDEX])
cotkg_sentences

['History of Present Illness: The patient is a 54-year-old female who presents with a complete absence of teeth.',
 'She reports having lost all her teeth over the years due to various dental issues.',
 'She has not had any dental prosthetics or implants.',
 'She has been managing without teeth but is now finding it increasingly difficult to eat and speak properly.',
 'She also reports a decrease in self-confidence due to her appearance.']

<hr style="height:0.1rem;">

### CoT ICD Code Semantic Search Knowledge Graph (KG)

Contains: 
- ICD Codes
- Patient Age
- Patient Gender
- Similar Clinical Example (10 cases) based on ICD Codes
- SNOMED-CT Ontology

In [27]:
### R006 means row #6 in the CSV (includes header row as Row 1)
COTSSKG_R006_GENERATED_TEXT = """History of Present Illness: The patient is a 54-year-old female who presents with a history of edentulism. She reports having lost all her teeth over the years due to various dental issues. She has been managing with dentures but has been experiencing discomfort and difficulty with chewing. She denies any recent trauma, infections, or significant weight loss. She has not noticed any changes in her sense of taste or any oral lesions. She does not smoke or drink alcohol. She has been maintaining good oral hygiene and regularly visits her dentist for check-ups. She is seeking consultation for possible dental implant options to improve her quality of life."""
COTSSKG_R033_GENERATED_TEXT = """The patient is a 54-year-old female who presents with a complete absence of teeth. She reports having lost her teeth over the years due to various dental issues and has not sought replacement options. She has been managing with a soft diet and has not experienced significant discomfort or difficulty with her current state of edentulism. However, she is now considering options for dental prosthetics for aesthetic reasons and to improve her ability to chew a wider variety of foods. She denies any current oral pain, swelling, or other symptoms. She has no known allergies and her medical history is unremarkable."""
COTSSKG_R051_GENERATED_TEXT = """The patient is a 54-year-old female who presents with a complete absence of teeth. She reports having lost her teeth over the years due to various dental issues. She has been managing with dentures but has been experiencing discomfort and difficulty with chewing. She denies any recent trauma, infections, or significant medical history. On physical examination, she is found to be completely edentulous with no signs of oral lesions or inflammation. She expresses interest in exploring options for dental implants."""
COTSSKG_R065_GENERATED_TEXT = """History of Present Illness: The patient is a 54-year-old female who presents with a history of edentulism. She reports having lost all her teeth over the years due to various dental issues. She has been managing with dentures but has been experiencing discomfort and difficulty in eating. She denies any recent trauma, infection, or significant weight loss. She has no history of oral cancer or any systemic diseases affecting the oral cavity. She does not smoke or consume alcohol. She is seeking consultation for possible dental implants for better comfort and functionality."""
COTSSKG_R078_GENERATED_TEXT = """The patient is a 54-year-old female who presents with a complete absence of teeth. She reports having lost her teeth over the years due to various dental issues and has not sought out dental prosthetics or implants. She denies any current oral pain or discomfort, but does express concern about her ability to eat certain foods and the impact on her overall appearance. She is interested in exploring options for restoring her dentition. Physical examination confirms edentulism, with no signs of oral infection or other abnormalities."""

# Add selected sentences to list
COTSSKG_GENERATED_TEXTS = [
    COTSSKG_R006_GENERATED_TEXT,
    COTSSKG_R033_GENERATED_TEXT,
    COTSSKG_R051_GENERATED_TEXT,
    COTSSKG_R065_GENERATED_TEXT,
    COTSSKG_R078_GENERATED_TEXT
]

# Convert CoT KG to sentences
cotsskg_sentences = sent_tokenize(COTSSKG_GENERATED_TEXTS[COTSSKG_GENERATED_TEXT_INDEX])
cotsskg_sentences

['History of Present Illness: The patient is a 54-year-old female who presents with a history of edentulism.',
 'She reports having lost all her teeth over the years due to various dental issues.',
 'She has been managing with dentures but has been experiencing discomfort and difficulty with chewing.',
 'She denies any recent trauma, infections, or significant weight loss.',
 'She has not noticed any changes in her sense of taste or any oral lesions.',
 'She does not smoke or drink alcohol.',
 'She has been maintaining good oral hygiene and regularly visits her dentist for check-ups.',
 'She is seeking consultation for possible dental implant options to improve her quality of life.']

<hr style="height:1rem;">

### Assessement

In [28]:
INDEX = 4

In [29]:
print("Sample Instance=", INDEX)

Sample Instance= 4


<hr style="height:0.1rem;">

### UMAP Settings

In [30]:
#n_neighbors = 10
#min_dist = 0.1
#random_state = 42

n_neighbors = 6
min_dist = 0.3
random_state = 1

n_neighbors = 3
min_dist = 0.1
random_state = 1

#n_neighbors = 15
#min_dist = 0.1
#random_state = 1


In [31]:
generated_texts = [
    sent_tokenize(BASELINE_GENERATED_TEXTS[INDEX]),
    sent_tokenize(COTSS_GENERATED_TEXTS[INDEX]),
    sent_tokenize(COTKG_GENERATED_TEXTS[INDEX]), 
    sent_tokenize(COTSSKG_GENERATED_TEXTS[INDEX])
]

sample_sizes = [
    len(sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES)),
    len(generated_texts[0]),
    len(generated_texts[1]),
    len(generated_texts[2]),
    len(generated_texts[3])
]
print('GT [samples]', sample_sizes[0])
print('Baseline [samples]', sample_sizes[1])
print('CoT ICD Code [samples]', sample_sizes[2])
print('CoT KG [samples]', sample_sizes[3])
print('CoT KG SS [samples]', sample_sizes[4])

n_samples = max(sample_sizes)
n_neighbors = min(15, n_samples - 1) if n_samples > 1 else 1
min_dist = 0.1
random_state = 1

print("n_neighbors=", n_neighbors)
print("min_dist=", min_dist)
print("random_state=", random_state)

GT [samples] 4
Baseline [samples] 8
CoT ICD Code [samples] 11
CoT KG [samples] 7
CoT KG SS [samples] 5
n_neighbors= 10
min_dist= 0.1
random_state= 1


In [ ]:
titles = [
    "Ground Truth vs Baseline",
    "Ground Truth vs CoT ICD Code",
    "Ground Truth vs CoT KG",
    "Ground Truth vs CoT KG SS"
]

plot_4_umap_subplots(
    sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES),
    generated_texts,
    titles,
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    random_state=random_state
)

In [ ]:
plot_4_umap_subplots1(
    sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES),
    generated_texts,
    titles,
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    random_state=random_state
)

In [ ]:
STOP ERROR

In [ ]:
plot_noun_similarity_heatmaps_4sets(
    sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES),
    sent_tokenize(BASELINE_GENERATED_TEXTS[INDEX]),
    sent_tokenize(COTSS_GENERATED_TEXTS[INDEX]),
    sent_tokenize(COTKG_GENERATED_TEXTS[INDEX]),
    sent_tokenize(COTSSKG_GENERATED_TEXTS[INDEX]),
    ['Baseline', 'CoT ICD Code', 'CoT KG', 'CoT KG SS']
)

<hr style="height:0.1rem;">

#### Baseline

In [ ]:
TITLE = "Ground Truth vs Baseline" 

In [ ]:
plot_noun_umap_interactive(
    sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES), 
    sent_tokenize(BASELINE_GENERATED_TEXTS[INDEX]), 
    title=TITLE,
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    random_state=random_state
)

In [ ]:
plot_noun_similarity_heatmap(
    sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES),
    sent_tokenize(BASELINE_GENERATED_TEXTS[INDEX]), 
    title=TITLE  
)

<hr style="height:0.1rem;">

#### CoT Semantic Search

In [ ]:
TITLE = "Ground Truth vs CoT ICD Code" 

In [ ]:
plot_noun_umap_interactive(
    sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES), 
    sent_tokenize(COTSS_GENERATED_TEXTS[INDEX]), 
    title=TITLE,
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    random_state=random_state
)

In [ ]:
plot_noun_similarity_heatmap(
    sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES),
    sent_tokenize(COTSS_GENERATED_TEXTS[INDEX]), 
    title=TITLE
)

<hr style="height:0.1rem;">

#### CoT KG

In [ ]:
TITLE = "Ground Truth vs CoT KG" 

In [ ]:
plot_noun_umap_interactive(
    sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES), 
    sent_tokenize(COTKG_GENERATED_TEXTS[INDEX]), 
    title=TITLE,
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    random_state=random_state
)

In [ ]:
plot_noun_similarity_heatmap(
    sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES),
    sent_tokenize(COTKG_GENERATED_TEXTS[INDEX]), 
    title=TITLE   
)

<hr style="height:0.1rem;">

#### CoT Semantic Search KG

In [ ]:
TITLE = "Ground Truth vs CoT KG SS" 

In [ ]:
plot_noun_umap_interactive(
    sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES), 
    sent_tokenize(COTSSKG_GENERATED_TEXTS[INDEX]), 
    title=TITLE,
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    random_state=random_state
)

In [ ]:
plot_noun_similarity_heatmap(
    sent_tokenize(CLINICAL_CASE_B_CLINICAL_NOTES),
    sent_tokenize(COTSSKG_GENERATED_TEXTS[INDEX]), 
    title=TITLE   
)